# Audit Log Configuration

**Table of contents**
- Overview

- Setup

  - Authentication Token

- Create Audit Log Configuration

  - Output Description

- Get Audit Log Configuration

  - Output Description

- Related Links

## Overview

This notebook demonstrates how to use the admin audit log configuration APIs to update and retrieve audit logging preferences for your organization.

These endpoints are intended for administrative users and allow you to:

- Update request and response logging configuration
- View the configuration set

The APIs require an admin token. If the authenticated user is not an admin, the service returns `403 Forbidden`.

## Setup

Ensure that Python 3+ is installed on your system.

<b>Note:</b> To run this notebook, add your credentials to `../../../auth/secrets.ini` and `../../../auth/config.ini`.

Example `secrets.ini` format:

```
[EAPI]
api.pat_token = <Your Emissions API PAT token>
api.tenant_id = <Your Emissions API Tenant Id>
```

In [ ]:
# Install the prerequisite Python packages
%pip install pandas configparser IPython requests

In [1]:
import pandas as pd
import configparser
import requests
import json
from IPython.display import display as display_summary

### Authentication Token

Run the following code snippet to generate a Bearer Token using your PAT token. These admin endpoints require an authenticated admin user.

In [ ]:
config = configparser.RawConfigParser()
config.read(['../../../auth/secrets.ini','../../../auth/config.ini'])

EAPI_PAT_TOKEN      = config.get('EAPI', 'api.pat_token')
EAPI_TENANT_ID      = config.get('EAPI', 'api.tenant_id')

EAPI_AUTH_ENDPOINT  = config.get('EAPI', 'api.auth_endpoint')
EAPI_BASE_URL       = config.get('EAPI', 'api.base_url')

EAPI_AUTH_CLIENT_ID = 'saascore-' + EAPI_TENANT_ID
EAPI_CLIENT_ID      = 'ghgemissions-' + EAPI_TENANT_ID

auth_request_headers: dict = {}
auth_request_headers["X-IBM-Client-Id"] = EAPI_AUTH_CLIENT_ID
auth_request_headers["X-IBM-Envizi-Pat"] = EAPI_PAT_TOKEN

verify = True

auth_url = f"{EAPI_AUTH_ENDPOINT}"
              
response = requests.post(url = auth_url,
                        headers = auth_request_headers,
                        verify  = verify
                       )
if response.status_code == 200:
    jwt_token = response.text
    print("Authentication Success")
else:     
    print("Authentication Failed")
    print(response.text)

Authentication Success


## Create Audit Log Configuration

Use `PUT /admin/audit-log` to update the audit logging preferences for the authenticated user's organization.

The request body accepts two boolean values:

- `logRequest`: Boolean value that enables or disables request payload logging.
- `logResponse`: Boolean value that enables or disables response payload logging.

In [11]:
PUT_AUDIT_LOG_ENDPOINT = f"{EAPI_BASE_URL}/admin/audit-log"

payload = {
    'logRequest': True,
    'logResponse': False
}

request_headers: dict = {}
request_headers['Content-Type'] = 'application/json'
request_headers['x-ibm-client-id'] = EAPI_CLIENT_ID
request_headers['Authorization'] = 'Bearer ' + jwt_token

response = requests.put(
    PUT_AUDIT_LOG_ENDPOINT,
    headers=request_headers,
    data=json.dumps(payload)
)

print(f'Status Code: {response.status_code}')

if response.text:
    response_json = response.json()
    display_summary(pd.json_normalize(response_json))
else:
    print('Empty Response')

Status Code: 200


,logRequest,logResponse,message
0,True,False,Audit log configuration created successfully


### Output Description

<b>logRequest</b> - Boolean flag indicating whether request payload logging is enabled.

<b>logResponse</b> - Boolean flag indicating whether response payload logging is enabled.

<b>message</b> - Response message. On success, the API returns: `Audit log configuration created successfully` or `Audit log configuration updated successfully`. If no changes were made, returns: `No change in audit log configuration`.

Possible HTTP status codes:

- `200` - Configuration updated successfully
- `400` - Invalid request parameters
- `403` - User is not authorized to access the admin endpoint
- `409` - No change in audit log configuration (conflict)
- `500` - Internal server error

## Get Audit Log Configuration

Use `GET /admin/audit-log` to retrieve the current audit logging configuration for the authenticated user's organization.

The response returns two boolean flags:

- `logRequest`: whether request payload logging is enabled
- `logResponse`: whether response payload logging is enabled

In [9]:
GET_AUDIT_LOG_ENDPOINT = f"{EAPI_BASE_URL}/admin/audit-log"

request_headers: dict = {}
request_headers['Content-Type'] = 'application/json'
request_headers['x-ibm-client-id'] = EAPI_CLIENT_ID
request_headers['Authorization'] = 'Bearer ' + jwt_token

response = requests.get(
    GET_AUDIT_LOG_ENDPOINT,
    headers=request_headers
)

print(f'Status Code: {response.status_code}')

if response.text:
    response_json = response.json()
    display_summary(pd.json_normalize(response_json))
else:
    print('Empty Response')

Status Code: 200


,logRequest,logResponse
0,True,False


### Output Description

<b>logRequest</b> - Boolean flag indicating whether incoming request bodies are logged for audit purposes.

<b>logResponse</b> - Boolean flag indicating whether outgoing response bodies are logged for audit purposes.

Possible error responses:

- `403` - User is not authorized to access the admin endpoint
- `404` - Audit log configuration was not found for the organization
- `400` - Bad request
- `500` - Internal server error

## Related Links

[Emissions API Developer Guide](https://developer.ibm.com/apis/catalog/ghgemissions--ibm-envizi-emissions-api/Introduction)